# V-Energy Grid 360 — perfil descritivo da Bronze

O notebook mede problemas por origem sem alterar os valores. Lê somente o
lote publicado, grava inventário por arquivo/aba e registra limites quando
uma coluna ou vínculo não pode ser verificado. Correções são decisão da Silver.

`tipo_bronze` descreve como cada coluna está armazenada no Delta. CSV e Excel ficam como texto; o JSON mantém a estrutura aninhada e os tipos físicos lidos. Isso não é uma inferência semântica de tipo.

## 0. Configuração e funções auxiliares

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, StructField, StructType
from datetime import datetime, timezone
import re
import pandas as pd

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("raw", "/Volumes/workspace/default/venergy")
dbutils.widgets.text("lote_origem", "")
CATALOG = dbutils.widgets.get("catalog").strip()
RAW = dbutils.widgets.get("raw").strip().rstrip("/")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG):
    raise ValueError("catalog deve ser um identificador simples do Unity Catalog")
if not RAW.startswith("/Volumes/"):
    raise ValueError("raw deve apontar para um Volume autorizado")
SCHEMA = "bronze"
SCHEMA_QUALIDADE = f"{CATALOG}.quality"
ALVO = f"{CATALOG}.{SCHEMA}"
VOLUME_RELATORIOS = f"{RAW}/relatorios"

TABELAS = [
    "unidades_consumidoras", "faturas", "leituras", "medidores",
    "ocorrencias_rede", "geracao_distribuida", "atendimentos", "inspecoes",
]
# Colunas que NÓS criamos na Bronze: ignoradas nas checagens. `_lote_id` é constante por execução,
# `_id_registro` é único por linha e `_ordem_na_origem` é a posição do registro no arquivo/aba —
# nenhuma delas é atributo do dado, então não entram em regra de qualidade (nulos, duplicatas,
# categorias, etc.). `_aba_origem` é a mesma coisa nas inspeções: registra de qual aba da planilha
# a linha veio, não é dado de negócio. Esquecer uma coluna de auditoria aqui tem efeito colateral:
# `_ordem_na_origem` é única por arquivo e faria a checagem de duplicatas exatas dar 0 sempre.
AUDITORIA = ["_arquivo_origem", "_aba_origem", "_lote_id", "_lote_origem", "_versao_origem", "_id_registro", "_ordem_na_origem", "_linha_fisica_inicial", "_linha_fisica_final", "_registro_bruto", "_data_carga"]

PROBLEMAS = []  # acumula o relatório final
AVISOS_ESQUEMA = set()  # regras que não puderam rodar por falta de coluna

_ativo = spark.sql(f"SELECT lote_ativo FROM {ALVO}.controle_publicacao WHERE chave = 'bronze'").first()
if not _ativo or not _ativo[0]:
    raise RuntimeError("Nenhum lote Bronze publicado")
LOTE_ATIVO = _ativo[0]
_status = spark.sql(f"SELECT status FROM {ALVO}.controle_lote WHERE _lote_id = '{LOTE_ATIVO}'").first()
if not _status or _status[0] != "PUBLICADO":
    raise RuntimeError(f"Lote {LOTE_ATIVO} ainda não confirmado como PUBLICADO")

def df(t):
    # Pina o lote no início do perfil para não misturar versões durante a análise.
    return spark.table(f"{CATALOG}.bronze_hist.{t}").where(F.col("_lote_id") == LOTE_ATIVO)

_COLS_BRUTAS = {}   # cache: as colunas de uma tabela não mudam durante a execução

def colunas_brutas(t):
    if t not in _COLS_BRUTAS:
        _COLS_BRUTAS[t] = df(t).columns
    return _COLS_BRUTAS[t]

def colunas(t):
    return [c for c in colunas_brutas(t) if c not in AUDITORIA]

def registrar_coluna_ausente(t, c):
    """Torna visível no relatório persistido uma regra que não pôde ser executada."""
    if (t, c) not in AVISOS_ESQUEMA:
        AVISOS_ESQUEMA.add((t, c))
        registra(t, f"regra não executada: coluna {c}", None, "NAO_EXECUTADA",
                 "Coluna ausente; ocorrências desconhecidas, não zero")
    print(f"  AVISO: coluna {t}.{c} não existe nesta Bronze — regra não executada.")


def existe(t, c):
    """Guarda contra mudança de esquema e registra a checagem omitida."""
    if c in colunas_brutas(t):
        return True
    registrar_coluna_ausente(t, c)
    return False

def vazio(c):
    """Considera nulo OU string vazia/espaços como 'faltante'."""
    return F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == "")

def num_br(c):
    """Número como ele vem nos arquivos: 'R$ 1.234,56' -> 1234.56 ; '217 min' -> 217 ; '6.31' -> 6.31.
    O que não vira número devolve nulo — é a mesma regra que a Silver aplica, então o perfil mede
    exatamente o que ela vai tratar."""
    s = F.regexp_replace(F.trim(F.col(c).cast("string")), r"[^0-9,.\-]", "")   # tira R$, 'min', letras
    br = F.regexp_replace(F.regexp_replace(s, r"\.", ""), ",", ".")           # 1.234,56 -> 1234.56
    limpo = F.when(F.instr(s, ",") > 0, br).otherwise(s)
    return F.when(limpo.rlike(r"^-?\d+(\.\d+)?$"), limpo).cast("double")

def data_br(c):
    """Lê as datas dos arquivos. O conteúdo mistura ISO com dia/mês primeiro e três separadores
    (- / .), e a ordem dia/mês NÃO é confiável: '10-19-2024' só pode ser mês primeiro, enquanto
    '27/07/2024' só pode ser dia primeiro. Depois de normalizar o separador, tentamos as três ordens.

    Devolve uma tupla:
      data    — primeiro formato que casa; nulo se nenhum casar;
      ambigua — casa nos DOIS sentidos e eles dão datas diferentes (ex.: '11-05-2022' pode ser
                11 de maio ou 5 de novembro). É uma decisão de negócio, não um valor inválido:
                o perfil registra, não esconde.

    O primeiro ramo usa `try_cast` de propósito: `ocorrencias_rede` e `atendimentos` guardam
    **timestamp** (`2025-07-21 14:07:00`), e o `cast` para data derruba a hora sem reclamar.
    Os outros dois ramos precisam de formato explícito, senão `11-05-2022` nunca casaria.
    """
    base = f"regexp_replace(trim(`{c}`), '[/.]', '-')"

    def parse(fmt):
        return F.expr(f"try_to_date({base}, '{fmt}')")

    iso = F.coalesce(F.expr(f"try_cast({base} as date)"),
                     F.expr(f"try_cast({base} as timestamp)").cast("date"))
    dia1, mes1 = parse("dd-MM-yyyy"), parse("MM-dd-yyyy")
    ambigua = dia1.isNotNull() & mes1.isNotNull() & (dia1 != mes1)
    return F.coalesce(iso, dia1, mes1), ambigua

def registra(tabela, verificacao, qtd, gravidade, comentario):
    """Guarda um problema; None significa que a regra não foi executada."""
    quantidade = None if qtd is None else int(qtd)
    PROBLEMAS.append({
        "tabela": tabela, "verificacao": verificacao, "ocorrencias": quantidade,
        "gravidade": gravidade, "observacao": comentario,
    })
    exibicao = "n/d" if quantidade is None else f"{quantidade:,}"
    print(f"[{gravidade:^14}] {tabela:<24} {verificacao:<45} {exibicao:>10}")

def regra(tabela, verificacao, necessarias, gravidade_base, comentario, conta):
    """Roda a checagem só se as colunas necessárias existirem (guarda de esquema) e registra o resultado.
    `conta` é uma função sem argumento: a consulta não roda nem é montada quando falta coluna."""
    faltando = [c for c in necessarias if c not in colunas_brutas(tabela)]
    if faltando:
        for coluna in faltando:
            registrar_coluna_ausente(tabela, coluna)
        print(f"  AVISO: '{verificacao}' ignorada — {tabela} sem: {', '.join(faltando)}")
        return
    q = conta()
    registra(tabela, verificacao, q, gravidade_base if q else "ok", comentario)

def mostra(pdf, titulo):
    """Exibe um resultado em pandas. Se estiver vazio, avisa em vez de tentar exibir (evita erro de schema vazio)."""
    print(f"\n--- {titulo} ---")
    if pdf is None or len(pdf) == 0:
        print("    (nada encontrado — nenhum problema deste tipo)")
    else:
        display(pdf)

def mostra_spark(d, titulo):
    """Exibe um DataFrame Spark. Se estiver vazio, avisa em vez de tentar exibir."""
    print(f"\n--- {titulo} ---")
    if d.limit(1).count() == 0:
        print("    (nada encontrado — nenhum problema deste tipo)")
    else:
        display(d)

print("Configuração ok. Tabelas:", len(TABELAS))

## 1. Estrutura das tabelas (schema)
Antes de checar qualidade, olhe **o que cada tabela tem**. Em especial `atendimentos`,
que veio **aninhado** do JSON — repare em campos que são `struct` ou `array`.

In [ ]:
for t in TABELAS:
    print(f"\n===== {t} =====")
    df(t).printSchema()

## 2. Visão geral
Quantas linhas e colunas cada tabela tem, e quantas **linhas idênticas** (duplicata exata).
Se `duplicatas_exatas` > 0, significa que a mesma linha aparece mais de uma vez no arquivo original.

In [ ]:
linhas = []
for t in TABELAS:
    d = df(t)
    total = d.count()
    # Duplicata EXATA = linhas iguais nos campos de DADO. Não usar dropDuplicates() sem argumento:
    # `_id_registro` é único por linha e faria a conta dar 0 duplicatas sempre (falso negativo).
    assinatura = F.sha2(F.to_json(F.struct(*[F.col("`" + c.replace("`", "``") + "`") for c in colunas(t)])), 256)
    dup = total - d.select(assinatura.alias("_assinatura")).distinct().count()
    linhas.append({"tabela": t, "linhas": total, "colunas": len(colunas(t)), "duplicatas_exatas": dup})
    # C1: entra no relatório consolidado (antes só aparecia nesta tabela de tela).
    registra(t, "linhas duplicadas (idênticas em todos os campos)", dup, "MEDIA" if dup else "ok",
             "Duplicatas apenas identificadas; decisão de tratamento pertence à Silver")

mostra(pd.DataFrame(linhas), "Visão geral")

## 3. Completude — valores nulos/vazios por coluna
Só lista colunas que têm algum valor faltante.
**Como interpretar:** um campo com 80% de nulo pode ser simplesmente "não se aplica"
(ex.: `data_retirada` de medidor ativo) — não é erro. O problema é nulo em campo **obrigatório**.

In [ ]:
nulos = []
for t in TABELAS:
    d = df(t)
    total = d.count()
    for c in colunas(t):
        n = d.where(vazio(c)).count()
        if n > 0:
            pct = round(100 * n / total, 2)
            nulos.append({"tabela": t, "coluna": c, "nulos": n, "percentual": pct})
            # C1: entra no relatório consolidado. Gravidade MEDIA porque nulo pode ser "não se aplica"
            # (ex.: data_retirada de medidor ativo) — quem decide se é campo obrigatório é o negócio.
            registra(t, f"nulos em {c}", n, "MEDIA",
                     f"{pct}% da tabela — conferir se o campo é obrigatório")

if nulos:
    tabela_nulos = pd.DataFrame(nulos).sort_values(["tabela", "percentual"], ascending=[True, False])
    mostra(tabela_nulos, "Nulos por coluna")
else:
    print("Nenhum valor nulo/vazio encontrado em nenhuma coluna.")

## 4. Unicidade — duplicatas na chave
Duplicata exata é rara; o problema comum é **a mesma chave repetida** (ex.: dois registros para o mesmo `id_uc`).
Chave repetida quebra joins e infla somas — é um dos erros que mais distorce o diagnóstico de receita.

In [ ]:
CHAVES = {
    "unidades_consumidoras": ["id_uc"],
    "faturas": ["id_fatura"],
    "leituras": ["id_leitura"],
    "medidores": ["id_medidor"],
    "ocorrencias_rede": ["id_ocorrencia"],
    "atendimentos": ["protocolo"],
    "inspecoes": ["id_inspecao"],
}

for t, ch in CHAVES.items():
    if not all(existe(t, c) for c in ch):
        continue
    d = df(t)
    chave_preenchida = F.lit(True)
    for coluna in ch:
        chave_preenchida = chave_preenchida & ~vazio(coluna)
    qtd = d.where(chave_preenchida).groupBy(*ch).count().where(F.col("count") > 1).count()
    registra(t, f"chave duplicada {ch}", qtd, "ALTA" if qtd else "ok",
             "Chave deveria ser única")

## 5. Integridade referencial — registros órfãos
Um "órfão" é um registro que aponta para algo que não existe.
Ex.: uma **leitura de UC que não está no cadastro** — se você fizer join na Silver, essa linha desaparece
silenciosamente (ou pior, infla um total). Por isso órfão vai para **quarentena**, não para o lixo.

In [ ]:
ucs_disponivel = existe("unidades_consumidoras", "id_uc")
ucs = (df("unidades_consumidoras").select("id_uc").distinct()
       if ucs_disponivel else None)

def orfaos(tabela, chave, referencia, rotulo, gravidade="ALTA"):
    """Conta chaves distintas e linhas afetadas sem alterar a Bronze."""
    d = df(tabela).where(~vazio(chave))
    referencia_valida = referencia.where(~vazio(chave))
    distintas = d.select(chave).distinct().join(referencia_valida, chave, "left_anti").count()
    linhas_afetadas = d.join(referencia_valida, chave, "left_anti").count()
    registra(tabela, rotulo, linhas_afetadas, gravidade if linhas_afetadas else "ok",
             f"{distintas:,} {chave}(s) distinta(s) em {linhas_afetadas:,} linha(s) afetada(s)")

def registrar_relacao_nao_executada(tabela, rotulo, motivo):
    registra(tabela, rotulo, None, "NAO_EXECUTADA", motivo)

for t in ["leituras", "faturas", "medidores", "ocorrencias_rede", "geracao_distribuida"]:
    fonte_tem_chave = existe(t, "id_uc")
    if fonte_tem_chave and ucs is not None:
        orfaos(t, "id_uc", ucs, "UC inexistente no cadastro (órfão)")
    elif fonte_tem_chave:
        registrar_relacao_nao_executada(
            t, "UC inexistente no cadastro (órfão)",
            "A coluna id_uc do cadastro de UCs está ausente")

# id_medidor das leituras x cadastro de medidores
medidores_tem_chave = existe("medidores", "id_medidor")
meds = (df("medidores").select("id_medidor").distinct()
        if medidores_tem_chave else None)
leituras_tem_medidor = existe("leituras", "id_medidor")
if leituras_tem_medidor and meds is not None:
    orfaos("leituras", "id_medidor", meds, "medidor inexistente no cadastro (órfão)")
elif leituras_tem_medidor:
    registrar_relacao_nao_executada(
        "leituras", "medidor inexistente no cadastro (órfão)",
        "A coluna id_medidor do cadastro de medidores está ausente")

# UCs de geração distribuída e ocorrências existentes sem leituras.
if ucs is not None and existe("leituras", "id_uc"):
    orfaos("unidades_consumidoras", "id_uc", df("leituras").select("id_uc").distinct(),
           "UC sem nenhuma leitura", gravidade="MEDIA")
else:
    registrar_relacao_nao_executada(
        "unidades_consumidoras", "UC sem nenhuma leitura",
        "Falta id_uc no cadastro ou nas leituras")


## 6. Validade — datas
Para cada coluna de data: mínimo, máximo e três contagens:
1. **inválida** — não casa com nenhum dos formatos do arquivo;
2. **ambígua** — casa nos dois sentidos (dia primeiro e mês primeiro) e dá datas diferentes,
   como `11-05-2022`; não é dado podre, é decisão de negócio pendente;
3. **no futuro** — sinal a investigar, não defeito provado.

O arquivo **não usa um formato só**: mistura ISO com dia/mês primeiro e três separadores (`- / .`).
Ler tudo como ISO — o que este notebook fazia — inflava "inválida" com data perfeitamente válida.

In [ ]:
DATAS = {
    "unidades_consumidoras": ["data_ligacao"],
    "faturas": ["data_emissao", "data_vencimento", "data_pagamento"],
    "leituras": ["data_leitura"],
    "medidores": ["data_instalacao", "data_retirada", "data_ultima_manutencao"],
    "ocorrencias_rede": ["data_abertura", "data_encerramento"],
    "geracao_distribuida": ["data_conexao"],
    "atendimentos": ["data_abertura", "data_fechamento"],
    "inspecoes": ["data"],
}

datas = []
for t, cols in DATAS.items():
    d = df(t)
    for c in cols:
        if not existe(t, c):
            continue
        p, amb = data_br(c)
        d2 = d.withColumn("_p", p).withColumn("_amb", amb)
        invalidas = d2.where(~vazio(c) & F.col("_p").isNull()).count()
        ambiguas = d2.where(F.col("_amb")).count()
        futuras = d2.where(F.col("_p") > F.current_date()).count()
        r = d2.where(F.col("_p").isNotNull()).agg(F.min("_p"), F.max("_p")).first()
        datas.append({"tabela": t, "coluna": c, "min": r[0], "max": r[1],
                      "invalidas": invalidas, "ambiguas": ambiguas, "no_futuro": futuras})
        registra(t, f"data inválida/impossível: {c}", invalidas,
                 "ALTA" if invalidas else "ok", "Não casa com nenhum formato conhecido")
        # Ambígua não é erro do dado: o arquivo simplesmente não diz a ordem dia/mês. Vai para o negócio.
        registra(t, f"data ambígua (dia/mês indefinido): {c}", ambiguas,
                 "MEDIA" if ambiguas else "ok",
                 "Casa nos dois sentidos; exige regra decidida, não chute")
        # C5: data no futuro é sinal a INVESTIGAR, não defeito provado — a competência corrente
        # pode legitimamente cair à frente da data da execução. Gravidade rebaixada.
        registra(t, f"data no futuro: {c} (investigar)", futuras,
                 "BAIXA" if futuras else "ok", "Pode ser competência corrente; conferir antes de tratar")
        if invalidas:
            print(f"  Exemplos de valores inválidos em {t}.{c}:")
            (d2.where(~vazio(c) & F.col("_p").isNull())
               .select(F.col(c).alias("valor_bruto"))
               .groupBy("valor_bruto").count()
               .orderBy(F.desc("count")).limit(10)
               .show(truncate=False))

mostra(pd.DataFrame(datas), "Datas")

## 7. Validade — números
Para cada coluna numérica: mínimo, máximo e quantos **negativos**.
Consumo negativo, por exemplo, é fisicamente impossível — é erro de leitura ou de cadastro.

In [ ]:
NUMERICAS = {
    "leituras": ["leitura_anterior", "leitura_atual", "consumo_kwh"],
    "faturas": ["valor_energia", "tributos", "valor_total"],
    "ocorrencias_rede": ["duracao_min"],
    "geracao_distribuida": ["potencia_kwp", "energia_injetada_kwh_media_mensal"],
    "unidades_consumidoras": ["latitude", "longitude"],
    "inspecoes": ["kwh_recuperado"],
    "atendimentos": ["nota_satisfacao"],
}

# C3: negativo vale para TODOS os campos numéricos, com duas exceções de julgamento:
#  - latitude/longitude: negativo é valor legítimo (hemisfério sul/oeste) — não é problema;
#  - valor_* de fatura: negativo pode ser estorno/ajuste — fica MEDIA, para conferir.
NEGATIVO_LEGITIMO = {"latitude", "longitude"}
NEGATIVO_IMPOSSIVEL = {"leitura_anterior", "leitura_atual", "consumo_kwh", "duracao_min",
                       "potencia_kwp", "energia_injetada_kwh_media_mensal",
                       "kwh_recuperado", "nota_satisfacao"}

numeros = []
for t, cols in NUMERICAS.items():
    d = df(t)
    for c in cols:
        if not existe(t, c):
            continue
        # C6: normaliza moeda/vírgula/sufixo antes de converter ('R$ 127,09' e '217 min' viram número).
        d2 = d.withColumn("_n", num_br(c))
        nao_num = d2.where(~vazio(c) & F.col("_n").isNull()).count()
        r = d2.where(F.col("_n").isNotNull()).agg(F.min("_n"), F.max("_n"), F.sum(F.when(F.col("_n") < 0, 1).otherwise(0))).first()
        negativos = int(r[2] or 0)
        numeros.append({"tabela": t, "coluna": c, "min": r[0], "max": r[1],
                        "negativos": negativos, "nao_numericos": nao_num})
        if nao_num:
            registra(t, f"valor não numérico: {c}", nao_num, "MEDIA", "Não converte para número")
            print(f"  Exemplos de valores não numéricos em {t}.{c}:")
            (d2.where(~vazio(c) & F.col("_n").isNull())
               .select(F.col(c).alias("valor_bruto"))
               .groupBy("valor_bruto").count()
               .orderBy(F.desc("count")).limit(10)
               .show(truncate=False))
        if negativos and c not in NEGATIVO_LEGITIMO:
            impossivel = c in NEGATIVO_IMPOSSIVEL
            registra(t, f"valor negativo: {c}", negativos, "ALTA" if impossivel else "MEDIA",
                     "Negativo fisicamente impossível" if impossivel
                     else "Verificar se é estorno/ajuste legítimo antes de tratar")

mostra(pd.DataFrame(numeros), "Números")

## 8. Consistência — categorias e variações de grafia
Aqui olhamos colunas de texto categóricas (município, bairro, classe, status...).
A segunda parte é a mais valiosa: **agrupa valores que são o mesmo texto escrito diferente**
(maiúsculas, espaços a mais). É o que precisa ser padronizado na Silver.

In [ ]:
CATEGORICAS = {
    "unidades_consumidoras": ["municipio", "bairro", "classe", "subgrupo_tarifario", "status"],
    "faturas": ["bandeira_tarifaria", "status"],
    "leituras": ["tipo_leitura"],
    "medidores": ["modelo", "tipo", "status"],
    "ocorrencias_rede": ["tipo"],
    "geracao_distribuida": ["modalidade"],
    "atendimentos": ["canal"],
    "inspecoes": ["resultado", "tipo_irregularidade"],
}

for t, cols in CATEGORICAS.items():
    for c in cols:
        if not existe(t, c):
            continue
        mostra_spark(df(t).where(~vazio(c)).groupBy(c).count().orderBy(F.desc("count")).limit(30),
                     f"{t}.{c}")

### 8.1. Mesmo valor, grafias diferentes
Agrupa pelo texto normalizado (**minúsculo, sem espaço extra**) e mostra os grupos com mais de uma
grafia. Roda em **todas** as colunas categóricas — não só em município e bairro.

Limite conhecido: a normalização **não** remove acento nem hífen, então `eletrônico` não colapsa
com `eletronico`. O que aparece aqui é o mínimo de padronização que os dados exigem, não o
inventário completo.

In [ ]:
def variantes(tabela, coluna):
    """Agrupa pelo texto normalizado e mostra grupos com mais de uma grafia (problema de padronização)."""
    d = df(tabela).where(~vazio(coluna)).select(F.col(coluna).alias("bruto"))
    norm = F.lower(F.regexp_replace(F.trim(F.col("bruto")), r"\s+", " "))
    g = (d.withColumn("normalizado", norm)
           .groupBy("normalizado")
           .agg(F.collect_set("bruto").alias("_v"), F.count("*").alias("linhas"))
           .where(F.size("_v") > 1)
           .withColumn("variantes", F.concat_ws(" | ", F.array_sort("_v")))
           .select("normalizado", "variantes", "linhas")
           .orderBy(F.desc("linhas")))
    return g

# Roda apenas nas colunas categóricas sem identificação pessoal. Antes eram só municipio e bairro — e o primeiro run real
# mostrou `classe`, `medidores.tipo` e `medidores.modelo` com variantes que nunca chegavam ao relatório.
for t, cols in CATEGORICAS.items():
    for c in cols:
        if not existe(t, c):
            continue
        v = variantes(t, c)
        qtd = v.count()
        registra(t, f"grafias diferentes em {c}", qtd, "MEDIA" if qtd else "ok",
                 "Mesmo valor escrito de formas diferentes")
        if qtd:
            # Já sabemos que não está vazio: evita o `limit(1).count()` extra do mostra_spark
            # (são 20 colunas categóricas × uma execução a mais em cada uma).
            print(f"\n--- Variantes de {t}.{c} ---")
            display(v)

## 9. Regras de negócio
Aqui entram checagens que só fazem sentido para o setor de energia.

In [ ]:
# --- Leituras ---
# Negativos não têm regra aqui: a seção 7 cobre TODOS os campos numéricos (item C3), com gravidade
# por coluna. Duplicar a regra aqui só encheria o relatório com a mesma linha duas vezes.

# leitura atual menor que a anterior (medidor pode virar/zerar — mas precisa de critério)
regra("leituras", "leitura_atual < leitura_anterior", ["leitura_atual", "leitura_anterior"], "MEDIA",
      "Pode ser troca de medidor ou erro de leitura",
      lambda: df("leituras").where(num_br("leitura_atual") < num_br("leitura_anterior")).count())

regra("leituras", "mes_referencia fora do formato AAAA-MM", ["mes_referencia"], "MEDIA",
      "Padrão de competência",
      lambda: df("leituras").where(~F.col("mes_referencia").rlike(r"^\d{4}-\d{2}$")).count())

# --- Faturas ---
regra("faturas", "valor_total != valor_energia + tributos", ["valor_total", "valor_energia", "tributos"], "ALTA",
      "Inconsistência de composição do valor",
      lambda: df("faturas").where(F.abs(num_br("valor_total") - (num_br("valor_energia") + num_br("tributos"))) > 0.01).count())

regra("faturas", "pagamento antes da emissão", ["data_pagamento", "data_emissao"], "ALTA",
      "Data de pagamento anterior à emissão da fatura",
      lambda: df("faturas").where(F.col("data_pagamento").isNotNull()
                                  & (data_br("data_pagamento")[0] < data_br("data_emissao")[0])).count())

regra("faturas", "status PAGA sem data_pagamento", ["status", "data_pagamento"], "MEDIA",
      "Status diz paga, mas não há data",
      lambda: df("faturas").where((F.upper(F.col("status")) == "PAGA") & vazio("data_pagamento")).count())

regra("faturas", "não paga com data_pagamento", ["status", "data_pagamento"], "MEDIA",
      "Data de pagamento preenchida mas status diferente",
      lambda: df("faturas").where((F.upper(F.col("status")) != "PAGA") & F.col("data_pagamento").isNotNull()).count())

regra("faturas", "mes_referencia fora do formato AAAA-MM", ["mes_referencia"], "MEDIA",
      "Padrão de competência",
      lambda: df("faturas").where(~F.col("mes_referencia").rlike(r"^\d{4}-\d{2}$")).count())

# --- Ocorrências ---
regra("ocorrencias_rede", "ocorrência sem data de encerramento", ["data_encerramento"], "MEDIA",
      "Interrupção em aberto",
      lambda: df("ocorrencias_rede").where(vazio("data_encerramento")).count())

regra("ocorrencias_rede", "encerrada mas sem duração", ["duracao_min", "data_encerramento"], "MEDIA",
      "Afeta o cálculo de DEC",
      lambda: df("ocorrencias_rede").where(vazio("duracao_min") & ~vazio("data_encerramento")).count())

# --- Cadastro ---
regra("unidades_consumidoras", "cpf_cnpj vazio", ["cpf_cnpj"], "MEDIA",
      "Campo sensível (LGPD) e possivelmente obrigatório",
      lambda: df("unidades_consumidoras").where(vazio("cpf_cnpj")).count())

regra("unidades_consumidoras", "status fora do domínio esperado", ["status"], "MEDIA",
      "Ver valores distintos na seção 8",
      lambda: df("unidades_consumidoras").where(~F.upper(F.col("status")).isin("ATIVA", "DESATIVADA", "SUSPENSA", "CORTADA")).count())

regra("unidades_consumidoras", "id_uc duplicado no cadastro", ["id_uc"], "ALTA",
      "Duas linhas para a mesma UC",
      lambda: df("unidades_consumidoras").select("id_uc").count() - df("unidades_consumidoras").select("id_uc").distinct().count())

## 10. Relatório consolidado de problemas
Esta é a saída principal do perfilamento. Ela vira:
- o **plano de limpeza** da camada Silver;
- a seção "decisões de limpeza" do **dicionário de dados**;
- o material para você justificar cada tratamento na apresentação.

In [ ]:
COLS_RELATORIO = ["tabela", "verificacao", "ocorrencias", "gravidade", "observacao"]
relatorio = pd.DataFrame(PROBLEMAS, columns=COLS_RELATORIO)
if len(relatorio) > 0:
    relatorio = relatorio.sort_values(["gravidade", "tabela"], ascending=[True, False])
    display(relatorio)
else:
    print("Nenhum problema registrado — dado surpreendentemente limpo. Revise as células anteriores.")

### 10.1. Persistência do relatório
O relatório é **insumo da Silver**, então precisa existir fora da tela: vai para a tabela Delta
`quality.perfil_bronze`, que passa a ser a **fonte da verdade** do plano de limpeza.

A gravação **não** fica dentro de `try/except`. Antes, uma falha de escrita era engolida com um
"o resultado está na tela" — e a Silver ficava sem insumo sem ninguém perceber. Se falhar agora,
a execução falha à vista.

In [ ]:
TABELA_RELATORIO = f"{SCHEMA_QUALIDADE}.perfil_bronze"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_QUALIDADE}")

ESQUEMA_RELATORIO = StructType([
    StructField("tabela", StringType(), False),
    StructField("verificacao", StringType(), False),
    StructField("ocorrencias", IntegerType(), True),
    StructField("gravidade", StringType(), False),
    StructField("observacao", StringType(), True),
])

linhas_relatorio = [
    (r["tabela"], r["verificacao"], r["ocorrencias"], r["gravidade"], r["observacao"])
    for r in PROBLEMAS
]
relatorio_spark = spark.createDataFrame(linhas_relatorio, schema=ESQUEMA_RELATORIO).withColumn(
    "_lote_id", F.lit(LOTE_ATIVO))

(relatorio_spark
   .withColumn("_data_perfilamento", F.lit(datetime.now(timezone.utc)).cast("timestamp"))
   .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable(TABELA_RELATORIO))
print(f"OK  {TABELA_RELATORIO}  {relatorio_spark.count():,} linhas")
if AVISOS_ESQUEMA:
    print(f"PERFIL INCOMPLETO: {len(AVISOS_ESQUEMA)} coluna(s) ausente(s); "
          "veja NAO_EXECUTADA em quality.perfil_bronze.")

# Cópia em CSV no volume, só para baixar e anexar na documentação. A persistência garantida é a tabela
# Delta acima; se o volume não existir, o aviso diz isso — em vez de fingir que salvou.
try:
    (relatorio_spark.coalesce(1).write.mode("overwrite").option("header", True)
       .csv(f"{VOLUME_RELATORIOS}/perfilamento_bronze"))
    print(f"CSV em {VOLUME_RELATORIOS}/perfilamento_bronze")
except Exception as e:
    print(f"CSV não gerado ({e}). O relatório está em {TABELA_RELATORIO}.")

### 10.2. Inventário das abas e mapa de relações

O nome de cada aba de inspeções indica um período aparente; a finalidade específica da separação depende de confirmação da origem. O inventário mede o grão observado por id_inspecao em cada aba. O mapa registra chaves compartilhadas, órfãos, máximo de linhas por chave e vigência temporal ainda não comprovada. FK_CANDIDATA é uma medição exploratória, não uma relação de negócio aprovada. COMPARACAO_DE_GRAO compara UC/competência entre faturas e leituras, sem tratar a ausência como fatura órfã. NAO_EXECUTADA e SEM_CHAVE_VERIFICADA mostram os vínculos que os arquivos não permitem provar. O perfil também conta competência fora de AAAA-MM, chaves repetidas e linhas excedentes, linhas CSV iguais ao cabeçalho inteiro e grupos de grafias candidatas em inspetor. Nomes de pessoas não são gravados nos achados; a padronização do inspetor depende de confirmar se o nome é exibido ao usuário. Nenhum desses achados altera a Bronze.


#### 10.2.1 Preparação do inventário



In [ ]:
# Inventário por origem, com contagem de registros e colunas reais.
from pyspark.sql.types import LongType


def persistir_lote(nome, linhas, schema):
    destino = f"{SCHEMA_QUALIDADE}.{nome}"
    d = spark.createDataFrame([tuple(row[field.name] for field in schema.fields)
                               for row in linhas], schema)
    if spark.catalog.tableExists(destino):
        spark.sql(f"DELETE FROM {destino} WHERE _lote_id = '{LOTE_ATIVO}'")
    (d.write.format("delta").mode("append").option("mergeSchema", "true")
        .saveAsTable(destino))
    print(f"OK {destino}: {len(linhas)} linhas do lote {LOTE_ATIVO}")


def origens_da_tabela(tabela):
    d = df(tabela)
    keys = [(arquivo, aba) for tab, arquivo, aba in reconc if tab == tabela]
    for arquivo, aba in keys:
        parte = d.where(F.col("_arquivo_origem") == arquivo)
        if aba is not None:
            parte = parte.where(F.col("_aba_origem") == aba)
        yield arquivo, aba, parte


def caminhos_schema(schema, prefixo=""):
    caminhos = []
    for field in schema.fields:
        if field.name in AUDITORIA:
            continue
        name = f"{prefixo}.{field.name}" if prefixo else field.name
        caminhos.append(name)
        if hasattr(field.dataType, "fields"):
            caminhos.extend(caminhos_schema(field.dataType, name))
        elif hasattr(field.dataType, "elementType") and hasattr(field.dataType.elementType, "fields"):
            caminhos.extend(caminhos_schema(field.dataType.elementType, name + "[]"))
    return caminhos


arquivos = {r["caminho"]: r.asDict() for r in
            spark.table(f"{ALVO}.arquivos_lote").where(F.col("_lote_id") == LOTE_ATIVO).collect()}
reconc = {(r["_tabela"], r["_arquivo_origem"], r["_aba_origem"]): r.asDict()
          for r in spark.table(f"{CATALOG}.bronze_hist.reconciliacao")
              .where(F.col("_lote_id") == LOTE_ATIVO).collect()}
inventario = []
relacoes = []
achados = []
distribuicoes = []
PERIODOS_ABAS = {
    "ago-set": "agosto a setembro",
    "out-nov": "outubro a novembro",
    "dez": "dezembro",
}
# O rótulo informa o período aparente, não prova que todas as datas pertençam a ele.
# Sem fonte de clientes separada ou FK observada, o vínculo fica explicitamente pendente.
LACUNAS_RELACIONAMENTO = {
    "unidades_consumidoras": [("cliente", "Sem fonte de clientes separada ou chave verificada")],
    "geracao_distribuida": [("faturas", "Potência cadastrada não comprova abatimento por competência")],
}
CATEGORIAS_SEGURAS = {
    "unidades_consumidoras": ["classe", "subgrupo_tarifario", "status"],
    "leituras": ["tipo_leitura"],
    "faturas": ["bandeira_tarifaria", "status"],
    "medidores": ["tipo", "status"],
    "ocorrencias_rede": ["tipo"],
    "geracao_distribuida": ["modalidade"],
    "atendimentos": ["canal", "status"],
    "inspecoes": ["resultado", "tipo_irregularidade"],
}


def achado(tabela, arquivo, aba, verificacao, coluna, quantidade, status="MEDIDA", detalhe=None):
    achados.append(dict(_lote_id=LOTE_ATIVO, tabela=tabela, arquivo=arquivo,
                        aba=aba, verificacao=verificacao, coluna=coluna,
                        ocorrencias=quantidade, status=status, detalhe=detalhe))


def resumir_padrao(parte, coluna, regras):
    """Conta padrões sem guardar exemplos ou valores brutos no relatório."""
    nome = "`" + coluna.replace("`", "``") + "`"
    bruto = F.trim(F.col(nome).cast("string"))
    categoria = F.when(bruto.isNull() | (bruto == ""), "VAZIO")
    for rotulo, regex in regras:
        categoria = categoria.when(bruto.rlike(regex), rotulo)
    categoria = categoria.otherwise("OUTRO")
    return {r["_padrao"]: int(r["count"]) for r in
            parte.select(categoria.alias("_padrao")).groupBy("_padrao").count().collect()}


def registrar_formatos_observados(parte, tabela, arquivo, aba):
    """Persiste padrões por origem; pontuação numérica não decide decimal versus milhar."""
    regras_data = [
        ("ISO_TIMESTAMP_FUSO", r"^\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}:\d{2}(?:\.\d+)?(?:Z|[+-]\d{2}:?\d{2})$"),
        ("ISO_TIMESTAMP", r"^\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}:\d{2}(?:\.\d+)?$"),
        ("ISO_DATE", r"^\d{4}-\d{2}-\d{2}$"),
        ("NUMERIC_DATE_SLASH", r"^\d{1,2}/\d{1,2}/\d{2,4}$"),
        ("NUMERIC_DATE_HYPHEN", r"^\d{1,2}-\d{1,2}-\d{2,4}$"),
        ("NUMERIC_DATE_DOT", r"^\d{1,2}\.\d{1,2}\.\d{2,4}$"),
    ]
    regras_fuso = [
        ("Z", r".*[Zz]$"),
        ("OFFSET", r".*[+-]\d{2}:?\d{2}$"),
        ("TIMESTAMP_SEM_FUSO", r".*[ T]\d{2}:\d{2}:\d{2}(?:\.\d+)?$"),
        ("DATA_SEM_TIMESTAMP", r"^\d{4}-\d{2}-\d{2}$"),
    ]
    regras_numericas = [
        ("PONTO_E_VIRGULA", r"(?=.*\.)(?=.*,)"),
        ("VIRGULA", r","),
        ("PONTO", r"\."),
        ("SEM_PONTUACAO", r"^[+-]?\d+$"),
    ]
    for coluna in DATAS.get(tabela, []):
        if coluna not in parte.columns:
            continue
        formatos = resumir_padrao(parte, coluna, regras_data)
        fusos = resumir_padrao(parte, coluna, regras_fuso)
        resumo = "; ".join(f"{k}={v}" for k, v in sorted(formatos.items()))
        achado(tabela, arquivo, aba, "formatos de data observados", coluna,
               sum(v for k, v in formatos.items() if k != "VAZIO"),
               "INVENTARIO", resumo or "Sem valores observáveis")
        resumo_fuso = "; ".join(f"{k}={v}" for k, v in sorted(fusos.items()))
        achado(tabela, arquivo, aba, "formatos de fuso observados", coluna,
               sum(v for k, v in fusos.items() if k not in ("VAZIO", "DATA_SEM_TIMESTAMP")),
               "INVENTARIO", resumo_fuso or "Sem valores observáveis")
    for coluna in NUMERICAS.get(tabela, []):
        if coluna not in parte.columns:
            continue
        formatos = resumir_padrao(parte, coluna, regras_numericas)
        resumo = "; ".join(f"{k}={v}" for k, v in sorted(formatos.items()))
        achado(tabela, arquivo, aba, "pontuação em valores numéricos observados", coluna,
               sum(v for k, v in formatos.items() if k != "VAZIO"),
               "INVENTARIO",
               (resumo + "; separador decimal/milhar não é inferido só pela pontuação")
               if resumo else "Sem valores observáveis")


def contar_variantes_inspetor(parte, tabela, arquivo, aba):
    """Conta candidatos a grafia variante sem persistir ou exibir nomes."""
    if "inspetor" not in parte.columns:
        achado(tabela, arquivo, aba, "grafias de inspetor", "inspetor",
               None, "NAO_EXECUTADA", "Coluna ausente")
        return
    bruto = F.col("inspetor")
    padrao = F.translate(
        F.upper(F.trim(F.regexp_replace(bruto, r"\s+", " "))),
        "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ",
        "AAAAAEEEEIIIIOOOOOUUUUC",
    )
    resumo = (parte.where(~vazio("inspetor"))
              .select(bruto.alias("_bruto"), padrao.alias("_normalizado"))
              .groupBy("_normalizado")
              .agg(F.countDistinct("_bruto").alias("_grafias"),
                   F.count("*").alias("_linhas"))
              .where(F.col("_grafias") > 1)
              .agg(F.count("*").alias("_grupos"),
                   F.sum("_linhas").alias("_linhas_afetadas"))
              .first())
    achado(tabela, arquivo, aba, "grafias de inspetor", "inspetor",
           int(resumo["_grupos"] or 0), "INVENTARIO",
           f"{int(resumo['_linhas_afetadas'] or 0)} linhas em grupos candidatos; "
           "confirmar uso do nome antes de alterar a Silver")


def checar_regra(parte, tabela, arquivo, aba, nome, colunas, condicao):
    faltantes = [c for c in colunas if c not in parte.columns]
    if faltantes:
        achado(tabela, arquivo, aba, nome, ", ".join(colunas), None,
               "NAO_EXECUTADA", "Colunas ausentes: " + ", ".join(faltantes))
    else:
        achado(tabela, arquivo, aba, nome, ", ".join(colunas),
               parte.where(condicao()).count())


#### 10.2.2 Medição por origem e mapa de relações



In [ ]:
for tabela in TABELAS:
    for arquivo, aba, parte in origens_da_tabela(tabela):
        total = parte.count()
        registrar_formatos_observados(parte, tabela, arquivo, aba)
        fonte = arquivos.get(arquivo, {})
        conta = reconc.get((tabela, arquivo, aba), {})
        dados = [field for field in parte.schema.fields if field.name not in AUDITORIA]
        if not dados:
            raise RuntimeError(f"{tabela}/{arquivo}/{aba}: nenhuma coluna da origem")
        achado(tabela, arquivo, aba, "falhas de parse", None,
               int(conta.get("rejeitadas") or 0))
        achado(tabela, arquivo, aba, "duplicatas exatas", None,
               int(conta.get("duplicatas_detectadas") or 0))
        if arquivo.endswith(".csv"):
            # Só uma linha igual ao cabeçalho inteiro é contada como repetição.
            cabecalho_repetido = F.lit(True)
            for field in dados:
                nome = field.name
                cabecalho_repetido = cabecalho_repetido & (
                    F.col("`" + nome.replace("`", "``") + "`").cast("string") == F.lit(nome))
            achado(tabela, arquivo, aba, "cabeçalho repetido", None,
                   parte.where(cabecalho_repetido).count(), "INVENTARIO",
                   "Igualdade exata de todas as colunas ao cabeçalho do CSV")
        if tabela == "inspecoes":
            contar_variantes_inspetor(parte, tabela, arquivo, aba)
        exprs = []
        for field in dados:
            col = F.col("`" + field.name.replace("`", "``") + "`")
            bruto = F.to_json(col) if hasattr(field.dataType, "fields") or hasattr(field.dataType, "elementType") else col.cast("string")
            exprs.extend([
                F.sum(F.when(col.isNull() | (F.trim(bruto) == ""), 1).otherwise(0)).alias(f"n_{len(exprs)}"),
                F.approx_count_distinct(bruto).alias(f"d_{len(exprs)}"),
            ])
        stats = parte.agg(*exprs).first()
        periodo_col = ("mes_referencia" if "mes_referencia" in parte.columns else
                       next((c for c in DATAS.get(tabela, []) if c in parte.columns), None))
        if periodo_col:
            periodo = (F.col(periodo_col) if periodo_col == "mes_referencia"
                       else data_br(periodo_col)[0])
            faixa = parte.select(periodo.alias("periodo")).agg(
                F.min("periodo"), F.max("periodo")).first()
            periodo_min, periodo_max = (str(faixa[0]) if faixa[0] is not None else None,
                                        str(faixa[1]) if faixa[1] is not None else None)
        else:
            periodo_min = periodo_max = None
        finalidade_aba = grao_aba = None
        if aba is not None:
            rotulo = PERIODOS_ABAS.get(aba, f"período indicado pelo nome: {aba}")
            finalidade_aba = (
                f"Inspeções de campo e resultados de fiscalização; aba rotulada {rotulo}. "
                "Finalidade específica da separação entre abas pendente de confirmação da origem."
            )
            if "id_inspecao" in parte.columns and total:
                ids_validos = parte.where(~vazio("id_inspecao"))
                sem_id = total - ids_validos.count()
                ids_repetidos = (ids_validos.groupBy("id_inspecao").count()
                                 .where(F.col("count") > 1).count())
                grao_aba = (f"Linha da aba; id_inspecao ausente em {sem_id} linhas e "
                            f"repetido em {ids_repetidos} chaves")
                if not sem_id and not ids_repetidos:
                    grao_aba = "Uma linha por id_inspecao nesta aba (unicidade observada)"
                achado(tabela, arquivo, aba, "grão da aba", "id_inspecao",
                       sem_id + ids_repetidos, "INVENTARIO", grao_aba)
            else:
                grao_aba = "Linha da aba; id_inspecao ausente ou aba vazia; grão de negócio não comprovado"
                achado(tabela, arquivo, aba, "grão da aba", "id_inspecao",
                       None, "NAO_EXECUTADA", grao_aba)
        for i, field in enumerate(dados):
            inventario.append(dict(
                _lote_id=LOTE_ATIVO, tabela=tabela, arquivo=arquivo, aba=aba,
                coluna=field.name, tipo_bronze=field.dataType.simpleString(),
                registros=total, nulos=int(stats[2*i] or 0),
                distintos_aproximados=int(stats[2*i+1] or 0),
                tamanho_bytes=fonte.get("tamanho_bytes"),
                hash_sha256=fonte.get("hash_sha256"),
                linhas_fisicas=conta.get("linhas_arquivo"),
                registros_logicos=conta.get("linhas_dados"),
                periodo_min=periodo_min, periodo_max=periodo_max,
                encoding="UTF-8 (validado na leitura)" if arquivo.endswith((".csv", ".json")) else None,
                delimitador=(";" if arquivo == "faturas.csv" else ","
                             if arquivo.endswith(".csv") else None),
                caminhos_json=(", ".join(caminhos_schema(parte.schema))
                               if tabela == "atendimentos" else None),
                finalidade_aba=finalidade_aba, grao_aba=grao_aba,
            ))
        for coluna in CHAVES.get(tabela, []):
            if coluna not in parte.columns:
                achado(tabela, arquivo, aba, "chave duplicada", coluna, None,
                       "NAO_EXECUTADA", "Coluna ausente")
            else:
                repetidas = (parte.where(~vazio(coluna)).groupBy(coluna).count()
                             .where(F.col("count") > 1)
                             .agg(F.count("*").alias("chaves"),
                                  F.sum(F.col("count") - 1).alias("excedentes"))
                             .first())
                achado(tabela, arquivo, aba, "chave duplicada", coluna,
                       int(repetidas["chaves"] or 0), detalhe="Número de chaves repetidas")
                achado(tabela, arquivo, aba, "linhas excedentes por chave", coluna,
                       int(repetidas["excedentes"] or 0),
                       detalhe="Linhas além da primeira por chave preenchida")
        for coluna in DATAS.get(tabela, []):
            if coluna not in parte.columns:
                achado(tabela, arquivo, aba, "data inválida ou ambígua", coluna,
                       None, "NAO_EXECUTADA", "Coluna ausente")
                continue
            parseada, ambigua = data_br(coluna)
            tmp = parte.withColumn("_p", parseada).withColumn("_amb", ambigua)
            achado(tabela, arquivo, aba, "data inválida", coluna,
                   tmp.where(~vazio(coluna) & F.col("_p").isNull()).count())
            achado(tabela, arquivo, aba, "data ambígua", coluna,
                   tmp.where(F.col("_amb")).count(), detalhe="Decisão pendente na Silver")
            achado(tabela, arquivo, aba, "data futura", coluna,
                   tmp.where(F.col("_p") > F.current_date()).count(),
                   detalhe="Investigar; não corrigir na Bronze")
        for coluna in NUMERICAS.get(tabela, []):
            if coluna not in parte.columns:
                achado(tabela, arquivo, aba, "número inválido ou negativo", coluna,
                       None, "NAO_EXECUTADA", "Coluna ausente")
                continue
            tmp = parte.withColumn("_n", num_br(coluna))
            achado(tabela, arquivo, aba, "não numérico", coluna,
                   tmp.where(~vazio(coluna) & F.col("_n").isNull()).count())
            achado(tabela, arquivo, aba, "negativo", coluna,
                   tmp.where(F.col("_n") < 0).count(),
                   detalhe="Pode ser legítimo em estornos ou coordenadas")
        if tabela == "leituras":
            checar_regra(parte, tabela, arquivo, aba, "leitura decrescente",
                         ["leitura_atual", "leitura_anterior"],
                         lambda: num_br("leitura_atual") < num_br("leitura_anterior"))
            checar_regra(parte, tabela, arquivo, aba, "competência fora de AAAA-MM",
                         ["mes_referencia"],
                         lambda: ~F.col("mes_referencia").rlike(r"^\d{4}-\d{2}$"))
        if tabela == "faturas":
            checar_regra(parte, tabela, arquivo, aba, "valor fora da composição",
                         ["valor_total", "valor_energia", "tributos"],
                         lambda: F.abs(num_br("valor_total") - num_br("valor_energia")
                                       - num_br("tributos")) > 0.01)
        if tabela == "unidades_consumidoras":
            checar_regra(parte, tabela, arquivo, aba, "status fora do domínio observado",
                         ["status"],
                         lambda: ~F.upper(F.col("status")).isin(
                             "ATIVA", "DESATIVADA", "SUSPENSA", "CORTADA"))
        for coluna in CATEGORIAS_SEGURAS.get(tabela, []):
            if coluna not in parte.columns:
                achado(tabela, arquivo, aba, "distribuição categórica", coluna,
                       None, "NAO_EXECUTADA", "Coluna ausente")
                continue
            top = (parte.groupBy(coluna).count().orderBy(F.desc("count"))
                   .limit(30).collect())
            contadas = 0
            for linha in top:
                qtd = int(linha["count"])
                contadas += qtd
                distribuicoes.append(dict(_lote_id=LOTE_ATIVO, tabela=tabela,
                                          arquivo=arquivo, aba=aba, coluna=coluna,
                                          valor=None if linha[coluna] is None else str(linha[coluna]),
                                          ocorrencias=qtd, classe="VALOR"))
            if total > contadas:
                distribuicoes.append(dict(_lote_id=LOTE_ATIVO, tabela=tabela,
                                          arquivo=arquivo, aba=aba, coluna=coluna,
                                          valor=None, ocorrencias=total-contadas,
                                          classe="OUTROS_FORA_DO_TOP_30"))
        # Mapa de relações observado, sem filtrar nem corrigir a Bronze.
        vinculos = ([] if tabela == "unidades_consumidoras" else
                    [(("id_uc",), "unidades_consumidoras", "FK_CANDIDATA")])
        if tabela == "leituras":
            vinculos.append((("id_medidor",), "medidores", "FK_CANDIDATA"))
        if tabela == "faturas":
            vinculos.append((("id_uc", "mes_referencia"), "leituras", "COMPARACAO_DE_GRAO"))
        # Só investigar ligações diretas quando a mesma chave identificadora existir nas duas fontes.
        if tabela == "atendimentos":
            vinculos.append((("id_ocorrencia",), "ocorrencias_rede", "FK_CANDIDATA"))
        if tabela == "inspecoes":
            vinculos.extend([
                (("protocolo",), "atendimentos", "FK_CANDIDATA"),
                (("id_ocorrencia",), "ocorrencias_rede", "FK_CANDIDATA"),
            ])
        for chaves, destino, natureza in vinculos:
            nome_chave = ", ".join(chaves)
            referencia = df(destino)
            faltantes = [c for c in chaves if c not in parte.columns or c not in referencia.columns]
            if faltantes:
                relacoes.append(dict(_lote_id=LOTE_ATIVO, tabela=tabela,
                                     arquivo=arquivo, aba=aba, chave=nome_chave,
                                     referencia=destino, status="NAO_EXECUTADA",
                                     linhas_orfas=None, chaves_distintas_orfas=None,
                                     max_linhas_por_chave_origem=None,
                                     max_linhas_por_chave_referencia=None,
                                     vigencia_temporal="Não comprovada",
                                     observacao="Chave compartilhada ausente: " + ", ".join(faltantes)))
                continue
            origem_com_chave, destino_com_chave = parte, referencia
            for chave in chaves:
                origem_com_chave = origem_com_chave.where(~vazio(chave))
                destino_com_chave = destino_com_chave.where(~vazio(chave))
            ref = destino_com_chave.select(*chaves).distinct()
            orfas = origem_com_chave.join(ref, list(chaves), "left_anti")
            origem_max = (origem_com_chave.groupBy(*chaves).count()
                          .agg(F.max("count")).first()[0] or 0)
            destino_max = (destino_com_chave.groupBy(*chaves).count()
                           .agg(F.max("count")).first()[0] or 0)
            observacao = ("Comparação por UC/competência: ausência não é órfão de fatura; "
                          "o par pode ter vários registros de cada lado e multiplicar fatos"
                          if natureza == "COMPARACAO_DE_GRAO" else
                          "Chave compartilhada observada; confirmar identidade e cardinalidade de negócio")
            relacoes.append(dict(_lote_id=LOTE_ATIVO, tabela=tabela,
                                 arquivo=arquivo, aba=aba, chave=nome_chave,
                                 referencia=destino, status=natureza,
                                 linhas_orfas=orfas.count(),
                                 chaves_distintas_orfas=orfas.select(*chaves).distinct().count(),
                                 max_linhas_por_chave_origem=origem_max,
                                 max_linhas_por_chave_referencia=destino_max,
                                 vigencia_temporal="Não comprovada: conferir datas e histórico na Silver",
                                 observacao=observacao))
        for destino, motivo in LACUNAS_RELACIONAMENTO.get(tabela, []):
            relacoes.append(dict(_lote_id=LOTE_ATIVO, tabela=tabela,
                                 arquivo=arquivo, aba=aba, chave="não verificada",
                                 referencia=destino, status="SEM_CHAVE_VERIFICADA",
                                 linhas_orfas=None, chaves_distintas_orfas=None,
                                 max_linhas_por_chave_origem=None,
                                 max_linhas_por_chave_referencia=None,
                                 vigencia_temporal="Não comprovada", observacao=motivo))


#### 10.2.3 Persistência dos relatórios



In [ ]:
# Sobreposição entre arquivos anuais é sinal para investigação, não descarte.
leituras = df("leituras")
if all(c in leituras.columns for c in ("id_uc", "mes_referencia")):
    sobrepostas = (leituras.groupBy("id_uc", "mes_referencia")
                   .agg(F.countDistinct("_arquivo_origem").alias("fontes"))
                   .where(F.col("fontes") > 1)
                   .select("id_uc", "mes_referencia"))
    por_arquivo = {r["_arquivo_origem"]: r["count"] for r in
                   leituras.join(sobrepostas, ["id_uc", "mes_referencia"])
                           .groupBy("_arquivo_origem").count().collect()}
    for arquivo, aba, _ in origens_da_tabela("leituras"):
        achado("leituras", arquivo, aba, "UC/competência em outro arquivo anual",
               "id_uc, mes_referencia", por_arquivo.get(arquivo, 0),
               detalhe="Pode ser medidor ou fatura múltipla; não deduplicar na Bronze")
else:
    for arquivo, aba, _ in origens_da_tabela("leituras"):
        achado("leituras", arquivo, aba, "UC/competência em outro arquivo anual",
               "id_uc, mes_referencia", None, "NAO_EXECUTADA", "Chaves ausentes")

inventario_schema = StructType([
    StructField("_lote_id", StringType(), False),
    StructField("tabela", StringType(), False),
    StructField("arquivo", StringType(), False),
    StructField("aba", StringType(), True),
    StructField("coluna", StringType(), False),
    StructField("tipo_bronze", StringType(), False),
    StructField("registros", LongType(), False),
    StructField("nulos", LongType(), False),
    StructField("distintos_aproximados", LongType(), False),
    StructField("tamanho_bytes", LongType(), True),
    StructField("hash_sha256", StringType(), True),
    StructField("linhas_fisicas", LongType(), True),
    StructField("registros_logicos", LongType(), True),
    StructField("periodo_min", StringType(), True),
    StructField("periodo_max", StringType(), True),
    StructField("encoding", StringType(), True),
    StructField("delimitador", StringType(), True),
    StructField("caminhos_json", StringType(), True),
    StructField("finalidade_aba", StringType(), True),
    StructField("grao_aba", StringType(), True),
])
relacoes_schema = StructType([
    StructField("_lote_id", StringType(), False),
    StructField("tabela", StringType(), False),
    StructField("arquivo", StringType(), False),
    StructField("aba", StringType(), True),
    StructField("chave", StringType(), False),
    StructField("referencia", StringType(), False),
    StructField("status", StringType(), False),
    StructField("linhas_orfas", LongType(), True),
    StructField("chaves_distintas_orfas", LongType(), True),
    StructField("max_linhas_por_chave_origem", LongType(), True),
    StructField("max_linhas_por_chave_referencia", LongType(), True),
    StructField("vigencia_temporal", StringType(), True),
    StructField("observacao", StringType(), True),
])
persistir_lote("inventario_bronze", inventario, inventario_schema)
persistir_lote("relacoes_bronze", relacoes, relacoes_schema)
achados_schema = StructType([
    StructField("_lote_id", StringType(), False),
    StructField("tabela", StringType(), False),
    StructField("arquivo", StringType(), False),
    StructField("aba", StringType(), True),
    StructField("verificacao", StringType(), False),
    StructField("coluna", StringType(), True),
    StructField("ocorrencias", LongType(), True),
    StructField("status", StringType(), False),
    StructField("detalhe", StringType(), True),
])
persistir_lote("achados_bronze", achados, achados_schema)
distribuicoes_schema = StructType([
    StructField("_lote_id", StringType(), False),
    StructField("tabela", StringType(), False),
    StructField("arquivo", StringType(), False),
    StructField("aba", StringType(), True),
    StructField("coluna", StringType(), False),
    StructField("valor", StringType(), True),
    StructField("ocorrencias", LongType(), False),
    StructField("classe", StringType(), False),
])
persistir_lote("distribuicoes_bronze", distribuicoes, distribuicoes_schema)


## 11. Como decidir o que fazer com cada problema
O relatório acima é a **lista de sintomas**. A decisão de tratamento é sua (e vai para o dicionário):

| Problema | Decisão comum | Quando NÃO usar |
|---|---|---|
| Duplicata exata | Remover | Se a duplicata carrega informação (ex.: dois pagamentos), investigar |
| Órfão (UC inexistente) | Mover para tabela de **quarentena** | Nunca apagar sem rastro |
| Nulo em campo opcional | Manter como nulo | — |
| Nulo em campo obrigatório | Investigar com o negócio antes de imputar | Imputar média sem justificar |
| Data inválida | Quarentena ou corrigir com regra clara | Corrigir "no chute" |
| Data no futuro | **Investigar** (pode ser competência corrente) | Tratar como erro sem evidência |
| Data ambígua (`11-05-2022`) | **Decidir a regra com o negócio**; até lá, quarentena | Assumir dia ou mês "porque sempre foi assim" |
| Grafia divergente | Padronizar (ex.: Title Case) | — |
| Consumo negativo | Verificar se é troca de medidor antes de tratar | Tratar como furto sem evidência |

**Próximo passo:** com este relatório em mãos, montar a Silver tratando cada item,
registrando a decisão na coluna `observacao` do dicionário.